# 5. Inverse PINNs, Identifiability, and Held-Out Validation

**Owner:** @rohanagrawal28 (WP05)

In this section, we estimate unknown parameters of strong-Allee harvesting models from discrete, noisy population observations.
We investigate two central questions:
1. **Structural Identifiability:** Are model parameters uniquely determined by ideal observation trajectories?
2. **Inference & Generalization:** How accurately can parameters and held-out future states be predicted under measurement noise using conventional ODE least squares versus Physics-Informed Neural Networks (PINNs)?

<div style="border-left:4px solid #176b87;padding:10px"><strong>Reporting fixes integrated.</strong> PR #10 commit <code>00393cd</code> measures validation loss with selection disabled and records optimizer exceptions/exits. Read <code>optimizer['lbfgs_outcome']</code> alongside the legacy <code>status</code>: <code>converged</code> means the training phase completed, not that L-BFGS reached a convergence tolerance. The recorded final sweep has 54 iteration-limit exits and no reported optimizer errors. Direct forward/inverse physics-loss parity is checked in the integration tests.</div>

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Automatic Colab & Local path detection
if "google.colab" in sys.modules:
    for candidate in [
        Path.cwd(),
        Path("/content"),
        Path("/content/Rohan_RK45_Data_and_Code"),
        Path("/content/mm-allee-harvesting"),
    ]:
        if (candidate / "src/mm_allee").exists():
            if str(candidate / "src") not in sys.path:
                sys.path.insert(0, str(candidate / "src"))
            os.chdir(candidate)
            break
else:
    ROOT = next(
        (
            p
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "src/mm_allee").exists()
        ),
        None,
    )
    if ROOT is not None and str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))

from mm_allee.data import generate_noisy_observations
from mm_allee.inference import fit_ode_least_squares
from mm_allee.models import Parameters
from mm_allee.solver import solve_trajectory

print("Loaded shared modules successfully.")

## 5.1 Model B Structural Nonidentifiability and Counterexample

Under Model B, harvesting is proportional to population density ($h N$):
$$\frac{dN}{dt} = r N \left(1 - \frac{N}{K}\right)\left(\frac{N}{A} - 1\right) - h N$$

Expanding the cubic polynomial:
$$\frac{dN}{dt} = -(r + h) N + r\left(\frac{1}{A} + \frac{1}{K}\right) N^2 - \frac{r}{AK} N^3$$

Any trajectory observed at a single constant effort $h$ reveals at most **three** composite polynomial combinations:
$$c_1 = -(r + h), \quad c_2 = r\left(\frac{1}{A} + \frac{1}{K}\right) = \frac{r(A+K)}{AK}, \quad c_3 = -\frac{r}{AK}$$

Because there are 4 unknown physical parameters $(r, A, K, h)$ but only 3 observable coefficients, the system is **structurally nonidentifiable** from data at a single fixed effort.

In [ ]:
# Demonstrate the exact two-parameter-set counterexample:
p1 = Parameters(r=1.0, A=0.2, K=1.0, h=0.5)
p2 = Parameters(r=1.35, A=0.3, K=0.9, h=0.15)

c1_1, c2_1, c3_1 = -(p1.r + p1.h), p1.r * (1.0 / p1.A + 1.0 / p1.K), -p1.r / (p1.A * p1.K)
c1_2, c2_2, c3_2 = -(p2.r + p2.h), p2.r * (1.0 / p2.A + 1.0 / p2.K), -p2.r / (p2.A * p2.K)

print(f"Parameter Set 1: c1={c1_1:.2f}, c2={c2_1:.2f}, c3={c3_1:.2f}")
print(f"Parameter Set 2: c1={c1_2:.2f}, c2={c2_2:.2f}, c3={c3_2:.2f}")
assert np.isclose(c1_1, c1_2) and np.isclose(c2_1, c2_2) and np.isclose(c3_1, c3_2)

# Solve trajectories from three distinct initial states for both parameter sets
d1, d2 = p1.dimensionless(), p2.dimensionless()
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

for n0, color in zip([0.4, 0.7, 1.1], colors, strict=True):
    traj1 = solve_trajectory(
        "B", a=d1["a"], harvest=d1["e"], x0=n0 / p1.K, tau_end=p1.r * 4.0, n_points=81
    )
    traj2 = solve_trajectory(
        "B", a=d2["a"], harvest=d2["e"], x0=n0 / p2.K, tau_end=p2.r * 4.0, n_points=81
    )
    t1, y1 = traj1.tau / p1.r, traj1.x * p1.K
    t2, y2 = traj2.tau / p2.r, traj2.x * p2.K
    ax.plot(t1, y1, label=f"Set 1 (N0={n0})", color=color, lw=2)
    ax.plot(t2, y2, "--", label=f"Set 2 (N0={n0})", color="black", alpha=0.7)

ax.set_title(
    "Model B Identifiability Counterexample: Identical Trajectories from Different Parameters"
)
ax.set_xlabel("Time t")
ax.set_ylabel("Population N(t)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 5.2 Observations, Noise Generation, and Frozen 60/20/20 Splits

We construct noisy observations according to the contract:
$$N_{\text{obs}}(t_i) = N_{\text{clean}}(t_i) + \sigma \cdot N_{\text{ref}} \cdot \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, 1)$$
with fixed external scale $N_{\text{ref}} = 100.0$. Observations are never silently clipped. Each trajectory has **101 observations** on $t\in[0,8]$.

Splits are strictly partitioned into contiguous time windows:
- **Train (60%, 61 points, $t\le4.8$):** used exclusively for fitting and data loss.
- **Validation (20%, 20 points, $4.88\le t\le6.4$):** never in the gradient; reported, and optionally used for early stopping (switched **off** in the final setting, see 5.4).
- **Held-out Test (20%, 20 points, $6.48\le t\le8$):** completely frozen until final post-training evaluation.

In [ ]:
p_true = Parameters(r=0.7, K=100.0, A=20.0, H=10.0)
d_true = p_true.dimensionless()
ref_traj = solve_trajectory(
    "A", a=d_true["a"], harvest=d_true["q"], x0=45.0 / 100.0, tau_end=0.7 * 8.0, n_points=101
)
times = ref_traj.tau / 0.7
clean_pops = ref_traj.x * 100.0

obs_5pct = generate_noisy_observations(
    times,
    clean_pops,
    trajectory_id="a_regular",
    model="A",
    noise_sigma=0.05,
    noise_seed=11,
    n_ref=100.0,
)

idx_tr, idx_va, idx_te = (
    obs_5pct["train_indices"],
    obs_5pct["val_indices"],
    obs_5pct["test_indices"],
)
print(f"Train points: {len(idx_tr)} (0 to {times[idx_tr[-1]]:.2f})")
print(f"Val points:   {len(idx_va)} ({times[idx_va[0]]:.2f} to {times[idx_va[-1]]:.2f})")
print(f"Test points:  {len(idx_te)} ({times[idx_te[0]]:.2f} to {times[idx_te[-1]]:.2f})")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axvspan(0, times[idx_tr[-1]], color="blue", alpha=0.1, label="Train Split (60%)")
ax.axvspan(times[idx_tr[-1]], times[idx_va[-1]], color="green", alpha=0.1, label="Val Split (20%)")
ax.axvspan(times[idx_va[-1]], times[-1], color="red", alpha=0.1, label="Held-out Test (20%)")
ax.plot(times, clean_pops, "k-", lw=2, label="Clean Reference")
ax.scatter(
    times,
    obs_5pct["observed_population"],
    color="darkblue",
    s=20,
    alpha=0.6,
    label="Noisy Observations (5% noise)",
)
ax.set_title("Model A: Observation Splits and Noise Structure")
ax.set_xlabel("Time t")
ax.set_ylabel("Population N(t)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5.3 Conventional Nonlinear Least-Squares ODE Baseline

We fit unknown parameters using SciPy's Trust Region Reflective solver on training observations only, with known $K=100.0$ and bounds $r > 0, 0 < A < K, H \ge 0$.

In [ ]:
fit_res = fit_ode_least_squares(
    "A",
    observations=obs_5pct,
    known_conditions={"known_K": 100.0, "known_initial_population": 45.0},
    fit_params=["r", "A", "H"],
    initial_guess={"r": 1.0, "A": 30.0, "H": 5.0},
    n_ref=100.0,
)

print(f"ODE Fit Status: {fit_res.status} in {fit_res.wall_seconds:.3f}s")
print("Estimated Parameters:")
for p in ["r", "A", "H"]:
    true_v = getattr(p_true, p)
    est_v = fit_res.estimated_parameters[p]
    rel_err = abs(est_v - true_v) / true_v * 100
    print(f"  {p}: True = {true_v:.2f}, Estimated = {est_v:.4f} (Error: {rel_err:.2f}%)")

print("Held-out Test Performance:")
print(
    f"  RMSE: {fit_res.test_metrics['rmse']:.4f}, Relative L2: {fit_res.test_metrics['relative_l2']:.4f}"
)

## 5.4 Inverse PINN with Differentiable Constraints

We optimize network weights and physical parameters jointly, preserving gradients via:
$$r = \mathrm{softplus}(\tilde{r}) > 0, \quad A = K \cdot \sigma(\tilde{A}) \in (0, K), \quad H = \mathrm{softplus}(\tilde{H}) \ge 0$$
Training loss is evaluated **strictly on training observations** plus the physics residual at 300 collocation points. The network, the network learning rate, the collocation count and the Adam (1000) + L-BFGS (2500) budget are the **same shared setting as the forward problem** (tanh, two hidden layers of 32, learning rate $10^{-3}$); only $\lambda_{\rm phys}=0.1$, the parameter learning rate $10^{-3}$ and the validation option are inverse-specific. Validation-based early stopping is switched off because on a flat late-time window it can return an early, barely trained state (see 5.6). The demo below uses noise $\sigma=0.01$, where Model A regular is informative.

In [ ]:
try:
    import jax  # noqa: F401  (availability check)

    base = next(
        p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments/integration").exists()
    )
    sys.path.insert(0, str(base / "experiments/integration"))
    from shared_settings import crb_relative_sd

    from mm_allee.pinn.inverse import InversePINNTrainer, TrainableParameters
    from mm_allee.pinn.network import NetworkConfig

    sigma = 0.01
    obs_noisy = generate_noisy_observations(
        times,
        clean_pops,
        trajectory_id="a_regular",
        model="A",
        noise_sigma=sigma,
        noise_seed=11,
        n_ref=100.0,
    )
    known = {"known_initial_population": 45.0, "known_K": 100.0}
    ncfg = NetworkConfig(layers=(1, 32, 32, 1), activation="tanh", seed=1011)
    tp = TrainableParameters(
        "A",
        fit_params=["r", "A", "H"],
        known_conditions={"known_K": 100.0},
        initial_guesses={"r": 1.0, "A": 30.0, "H": 5.0},
    )
    trainer = InversePINNTrainer(
        network_config=ncfg,
        trainable_params=tp,
        lr_net=1e-3,
        lr_params=1e-3,
        lambda_data=1.0,
        lambda_phys=0.1,
        lbfgs_iters=2500,
        lbfgs_ftol=0.0,
        lbfgs_gtol=0.0,
        validation_selection=False,
    )
    pinn_res = trainer.fit(
        obs_noisy, known_conditions=known, epochs=1000, n_collocation=300, seed=1011
    )

    train_t = np.asarray(obs_noisy["time"])[np.asarray(obs_noisy["split"]) == "train"]
    bound = crb_relative_sd(
        "A",
        ["r", "A", "H"],
        {"known_K": 100.0, "known_initial_population": 45.0},
        {"r": 0.7, "K": 100.0, "A": 20.0, "H": 10.0, "h": 0.0},
        train_t,
        sigma * 100.0,
    )
    floor = pinn_res.train_metrics["rmse"] / (sigma * 100.0)
    print(f"PINN completed: {pinn_res.status} in {pinn_res.wall_seconds:.1f}s")
    print("L-BFGS exit:", pinn_res.optimizer["lbfgs_outcome"], pinn_res.optimizer["lbfgs_message"])
    print(f"Returned-model validation loss: {pinn_res.val_loss:.6g}")
    print(
        f"training RMSE / noise sd = {floor:.2f}  (about 0.9 means the fit reached the noise floor)"
    )
    for p in ["r", "A", "H"]:
        true_v = getattr(p_true, p)
        est_v = pinn_res.estimated_parameters[p]
        rel_err = abs(est_v - true_v) / true_v
        print(
            f"  {p}: true {true_v:.2f}, PINN {est_v:.4f}, relative error {rel_err:.1e}, "
            f"Cramer-Rao bound {bound[p]:.1e}"
        )
except ImportError:
    print(
        "JAX is not installed in this environment; skipping PINN training. Full code is implemented in mm_allee.pinn."
    )

## 5.5 Final sweep results (loaded from `artifacts/`, not retrained)

`python experiments/wp05/run.py --config configs/wp05_final.json --output results/wp05` runs 6 cases $\times$ 3 noise levels $\times$ 3 seeds = 54 PINN fits and 54 ODE fits. The table below counts the **clean-data** fits ($\sigma=0$) in which *every* fitted parameter is within $5\times10^{-2}$ relative error. Rohan's original settings (64x64, 120 points, validation early stopping, 200 L-BFGS iterations) recovered 0 of 18 and all 54 of their fits ended `validation_stopped`.

**What `converged` means here:** only that the Adam phase finished and a finite solution was returned. In all 54 fits L-BFGS stopped at its iteration limit (no optimiser error, no abnormal exit, no tolerance reached; see the `lbfgs_outcome` column of `wp05_prediction_metrics.csv`); the 36 noisy fits are at the noise floor (training RMSE 0.65 to 1.15 times the noise sd).

In [ ]:
import csv  # noqa: F811  (an earlier fragment imports csv in the assembled notebook)
import statistics

rows = list(csv.DictReader(open("artifacts/tables/wp05_parameter_metrics.csv")))
fits = {}
for r in rows:
    if float(r["noise_sigma"]) == 0.0:
        fits.setdefault((r["method"], r["run_id"]), {})[r["parameter"]] = float(r["relative_error"])
cases = ["a_regular", "b_regular", "a_extinction", "b_near_fold", "a_near_fold", "b_decline"]
print(f"{'case':14s}{'PINN recovered':>16s}{'PINN median err':>17s}{'ODE_LS median err':>19s}")
total = 0
for c in cases:
    pinn = [max(v.values()) for (m, rid), v in fits.items() if m == "PINN" and f"wp05_{c}_" in rid]
    ode = [max(v.values()) for (m, rid), v in fits.items() if m == "ODE_LS" and f"wp05_{c}_" in rid]
    ok = sum(e <= 0.05 for e in pinn)
    total += ok
    print(
        f"{c:14s}{ok:>13d}/{len(pinn)}{statistics.median(pinn):>17.1e}{statistics.median(ode):>19.1e}"
    )
print(f"PINN total recovered (every parameter within 5e-2): {total}/18   (original settings: 0/18)")

![Parameter recovery versus noise](../artifacts/figures/wp05_recovery_noise.png)

![Held-out windows, Model A regular](../artifacts/figures/wp05_heldout_failure.png)

The first figure averages over all six cases and seeds, so its error bars are dominated by the badly conditioned cases. The second shows Model A regular (top row noise-free, bottom row noisy); the regular trajectories are at equilibrium by $t=4.8$, so the held-out window is an undemanding extrapolation test.

## 5.6 Why some cases were not reliably recovered, and whether noise is the reason

- **Poor conditioning.** Model A near-fold, Model B near-fold and Model B decline have Jacobian condition numbers of $10^{3}$ to $2\times10^{4}$ (62 to 340 for the recoverable cases). The PINN fits the training data to $10^{-4}$ to $10^{-5}$ while the parameters are off by 40% to 100%; the ODE fit reaches $10^{-7}$ to $10^{-10}$, so the problems are identifiable in principle. In the near-fold cases $r$ and $A$ collapse together to about $10^{-6}$ with $r/A$ fixed, a degenerate valley of the cubic. None of the settings tested fixed this on the seeds tried. These cases were not reliably recovered by the tested PINN settings and budgets, and poor conditioning magnifies small trajectory-fitting errors into large parameter errors; this is **not** the proven structural ambiguity of Model B with all four parameters unknown ($K$ is known here and the ODE fit recovers every case).
- **Noise is not the cause.** At $\sigma\in\{0,0.001,0.005,0.01\}$ (tanh and swish, 144 fits) the optimiser reaches the noise floor in 132 of 144 fits; well-conditioned errors are about 1.3 to 1.4 times the Cramér–Rao bound at every $\sigma$; the share of collapsed fits does not depend on $\sigma$.
- **The 12 fits that missed the noise floor** all came from validation-based early stopping (status `validation_stopped`, parameters still at the starting guess). Switched off, all 144 fits reach the noise floor, but recovery does not improve overall: the remaining recovery failures concern the poorly conditioned cases under the tested settings, not the proven four-unknown Model B ambiguity.
- **Judge noisy recovery against the Cramér–Rao bound.** At $\sigma=0.01$ only Model A and B regular are informative (bound at most 25%); there the shared setting is within twice the bound in 6 of 6 fits (original settings 0 of 6). At $\sigma=0.05$ no case is informative.

## 5.7 Key Scientific Takeaways

1. **Parameter Error $\ne$ Trajectory Error:** the PINN fits every case to about $10^{-4}$ or better but recovers the parameters only where the problem is well conditioned (10 of 18 clean fits), and under noise only as well as the Cramér–Rao bound allows.
2. **Classical ODE Baselines are Strong:** for scalar ODEs with low-dimensional parameter spaces, SciPy nonlinear least squares is faster and recovers all six cases on clean data.
3. **Structural Identifiability Precedes Optimization:** Model B with unknown effort $h$ is mathematically ambiguous; no optimization algorithm can break this ambiguity without additional independent information (e.g. known $K$ or multiple effort levels).
4. **Check convergence against the noise level:** a short L-BFGS budget with a tolerance stop and validation-based checkpoint selection silently returned near-initial parameters in every run of the original sweep; the ratio of training RMSE to noise sd exposes this.